In [ ]:
!pip install timm einops

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import timm
from einops import rearrange
import math
import copy

In [ ]:
class CFG:
    batch_size = 128
    lr = 3e-4
    epochs = 3
    img_size = 224
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    temperature = 0.2
    ema_decay = 0.996
    mask_ratio = 0.75

cfg = CFG()

## **Data**

In [ ]:
class MultiViewTransform:
    def __init__(self):
        self.base = T.Compose([
            T.RandomResizedCrop(32),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.4,0.4,0.4,0.1),
            T.ToTensor()
        ])

    def __call__(self, x):
        return [self.base(x), self.base(x)]

dataset = CIFAR10(root="./data", train=True, download=True, transform=MultiViewTransform())
loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=2, drop_last=True)

In [ ]:
class ViTBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True)
        self.vit.head = nn.Identity()

    def forward(self, x):
        x = F.interpolate(x, size=cfg.img_size)
        return self.vit(x)

## **Projection Head**

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=1024, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)

## **Predictor (BYOL)**

In [ ]:
class Predictor(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return self.net(x)

## **MAE Module**

In [ ]:
class MaskedAutoencoder(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = nn.Sequential(
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Linear(512, 768)
        )

    def random_mask(self, x, ratio):
        B, D = x.shape
        mask = torch.rand(B, D, device=x.device) < ratio
        x_masked = x.clone()
        x_masked[mask] = 0
        return x_masked, mask

    def forward(self, x):
        features = self.encoder(x)
        masked, mask = self.random_mask(features, cfg.mask_ratio)
        recon = self.decoder(masked)
        loss = F.mse_loss(recon[mask], features[mask])
        return loss

## **Main Model**

In [ ]:
class HybridSSLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ViTBackbone()
        self.projector = MLP(768)
        self.predictor = Predictor()

        # EMA teacher
        self.teacher = copy.deepcopy(self.encoder)
        for p in self.teacher.parameters():
            p.requires_grad = False

        self.mae = MaskedAutoencoder(self.encoder)

    def forward(self, x1, x2):
        # Student
        h1 = self.encoder(x1)
        h2 = self.encoder(x2)

        z1 = F.normalize(self.projector(h1), dim=1)
        z2 = F.normalize(self.projector(h2), dim=1)

        p1 = self.predictor(z1)
        p2 = self.predictor(z2)

        # Teacher (EMA)
        with torch.no_grad():
            t1 = F.normalize(self.projector(self.teacher(x1)), dim=1)
            t2 = F.normalize(self.projector(self.teacher(x2)), dim=1)

        return p1, p2, z1, z2, t1.detach(), t2.detach()

    def update_teacher(self):
        for param_q, param_k in zip(self.encoder.parameters(), self.teacher.parameters()):
            param_k.data = param_k.data * cfg.ema_decay + param_q.data * (1 - cfg.ema_decay)

## **Losses**

In [ ]:
def simclr_loss(z1, z2):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.t()) / cfg.temperature

    labels = torch.arange(B).to(cfg.device)
    labels = torch.cat([labels, labels], dim=0)

    return F.cross_entropy(sim, labels)

def byol_loss(p, z):
    return 2 - 2 * (p * z).sum(dim=-1).mean()

def dino_loss(student, teacher):
    return - (teacher * F.log_softmax(student, dim=-1)).sum(dim=-1).mean()

## **Train**

In [ ]:
model = HybridSSLModel().to(cfg.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

scaler = torch.cuda.amp.GradScaler()

for epoch in range(cfg.epochs):
    for (views, _) in loader:
        x1, x2 = views[0].to(cfg.device), views[1].to(cfg.device)

        with torch.cuda.amp.autocast():
            p1, p2, z1, z2, t1, t2 = model(x1, x2)

            loss_simclr = simclr_loss(z1, z2)
            loss_byol = byol_loss(p1, t2) + byol_loss(p2, t1)
            loss_dino = dino_loss(z1, t2) + dino_loss(z2, t1)
            loss_mae = model.mae(x1)

            loss = loss_simclr + loss_byol + loss_dino + loss_mae

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        model.update_teacher()

    print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

print("Pretraining Done")

## **Linear Probing**

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(768, 10)

        for p in self.encoder.parameters():
            p.requires_grad = False

    def forward(self, x):
        x = self.encoder(x)
        return self.fc(x)

probe = LinearProbe(model.encoder).to(cfg.device)
opt_probe = torch.optim.Adam(probe.fc.parameters(), lr=1e-3)

# Supervised dataset
transform = T.Compose([
    T.Resize((32,32)),
    T.ToTensor()
])
dataset_sup = CIFAR10(root="./data", train=True, download=True, transform=transform)
loader_sup = DataLoader(dataset_sup, batch_size=cfg.batch_size, shuffle=True)

for epoch in range(2):
    for x, y in loader_sup:
        x, y = x.to(cfg.device), y.to(cfg.device)

        logits = probe(x)
        loss = F.cross_entropy(logits, y)

        opt_probe.zero_grad()
        loss.backward()
        opt_probe.step()

    print(f"Probe Epoch {epoch} | Loss: {loss.item():.4f}")

print("Linear Probing Done")